In [ ]:
# Load existing vector store
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
# print("Loaded existing vector store!")

## 10. Load Existing Vector Store (Optional)

If you've already created a vector store, you can load it instead of recreating it.

In [ ]:
def ask_question(question):
    """Helper function to query the RAG system"""
    result = qa_chain.invoke({"query": question})
    print(f"Q: {question}")
    print(f"A: {result['result']}\n")
    return result

# Try multiple questions
questions = [
    "What is RAG?",
    "What are the benefits of using Ollama?",
    "Explain machine learning and deep learning"
]

for q in questions:
    ask_question(q)
    print("-" * 80)

## 9. Interactive Query Function

In [ ]:
# Ask a question
query = "What is Ollama and what can it do?"

result = qa_chain.invoke({"query": query})

print("Question:", query)
print("\nAnswer:", result['result'])
print("\n" + "="*50)
print("Source Documents:")
for i, doc in enumerate(result['source_documents'], 1):
    print(f"\n{i}. {doc.metadata['source']}")
    print(doc.page_content[:200] + "...")

## 8. Query the RAG System

In [ ]:
# Create a custom prompt template
template = """Use the following pieces of context to answer the question at the end. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}

Question: {question}

Answer:"""

QA_CHAIN_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)

# Create retrieval QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

print("RAG chain created successfully!")

## 7. Create RAG Chain

In [ ]:
# Initialize embedding model (runs locally)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create vector store using ChromaDB
vectorstore = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Vector store created successfully!")

## 6. Create Embeddings and Vector Store

In [ ]:
# Load documents from directory
loader = DirectoryLoader('documents/', glob="**/*.txt", loader_cls=TextLoader)
documents = loader.load()

print(f"Loaded {len(documents)} documents")

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
texts = text_splitter.split_documents(documents)

print(f"Split into {len(texts)} chunks")

## 5. Load and Split Documents

In [ ]:
# Create a documents directory
os.makedirs("documents", exist_ok=True)

# Create sample documents
sample_docs = {
    "documents/ai_basics.txt": """
Artificial Intelligence (AI) is the simulation of human intelligence by machines.
Machine learning is a subset of AI that enables systems to learn from data.
Deep learning uses neural networks with multiple layers to process complex patterns.
Natural language processing helps computers understand and generate human language.
""",
    "documents/ollama_info.txt": """
Ollama is a tool for running large language models locally on your machine.
It supports various models including Llama 2, Mistral, and CodeLlama.
Ollama makes it easy to run models without requiring cloud services.
You can customize models and use them offline for privacy and convenience.
""",
    "documents/rag_explanation.txt": """
Retrieval-Augmented Generation (RAG) combines information retrieval with text generation.
RAG systems first retrieve relevant documents from a knowledge base.
The retrieved context is then provided to the LLM to generate informed responses.
This approach reduces hallucinations and provides up-to-date information.
"""
}

for filename, content in sample_docs.items():
    with open(filename, 'w') as f:
        f.write(content)

print("Sample documents created successfully!")

## 4. Create Sample Documents

Create some sample text files to use for RAG, or load your own documents.

In [ ]:
# Initialize Ollama with your preferred model
llm = Ollama(
    model="mistral",  # or "llama2", "llama3", "codellama", etc.
    temperature=0.7
)

# Test the connection
print(llm.invoke("Hello! Respond with a brief greeting."))

## 3. Initialize Ollama LLM

Make sure Ollama is running and you have pulled a model (e.g., `ollama pull mistral`)

In [ ]:
from langchain_community.llms import Ollama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain.prompts import PromptTemplate
import os

## 2. Import Libraries

In [ ]:
!pip install langchain langchain-community chromadb sentence-transformers pypdf

## 1. Install Required Packages

# RAG with Ollama

This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) system using:
- **Ollama** for running local LLMs
- **LangChain** for RAG pipeline
- **ChromaDB** for vector storage
- **HuggingFace embeddings** for document embeddings
